# Hospital Readmission Analysis

## Objective
Explore hospital records of patients with diabetes and investigate
factors associated with readmission within 30 days.

## Dataset
Diabetes 130 US Hospitals for Years 1999–2008.

Source: https://www.kaggle.com/datasets/brandao/diabetes

## Initial exploration
- Inspect the dataset structure and column meanings.
- Identify missing values and duplicate records.
- Explore the distribution of readmission outcomes.

This project is for educational analysis.

In [4]:
from pathlib import Path

data_path = Path("../data/raw/diabetic_data.csv")
print("Dataset found:", data_path.exists())

Dataset found: True


In [11]:
import pandas as pd

# Recognize "?" as missing and preserve "None" labels
df = pd.read_csv(
    data_path,
    keep_default_na=False,
    na_values=["?"]
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 101766
Columns: 50


/var/folders/lw/dbrf54n571n8b9g__cjb5_6h0000gn/T/ipykernel_96229/2138529852.py:4: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [17]:
df = pd.read_csv(data_path, keep_default_na=False)

In [13]:
# Count missing values, including question-mark placeholders
missing = df.replace("?", pd.NA).isna().sum()

print("Columns with missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Count fully duplicated rows
print("\nDuplicate rows:", df.duplicated().sum())

Columns with missing values:
weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

Duplicate rows: 0


In [14]:
# Preserve original text labels to inspect missing-value codes
df = pd.read_csv(data_path, keep_default_na=False)

for column in ["weight", "max_glu_serum", "A1Cresult"]:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


weight:
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

max_glu_serum:
max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64

A1Cresult:
A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64


In [16]:
# Summarize missing values
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_percent", ascending=False)

missing_summary.round(2)

,missing_count,missing_percent
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [18]:
# Count readmission outcomes and calculate percentages
readmission_summary = pd.DataFrame({
    "count": df["readmitted"].value_counts(dropna=False),
    "percent": df["readmitted"].value_counts(
        normalize=True, dropna=False
    ) * 100
})

readmission_summary.round(2)

,count,percent
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16


In [19]:
# Distinguish hospital encounters from unique patients
print("Hospital encounters:", len(df))
print("Unique patients:", df["patient_nbr"].nunique())
print("Unique encounter IDs:", df["encounter_id"].nunique())

# Count patients with multiple encounters
encounters_per_patient = df.groupby("patient_nbr").size()
print("Patients with multiple encounters:",
      (encounters_per_patient > 1).sum())

Hospital encounters: 101766
Unique patients: 71518
Unique encounter IDs: 101766
Patients with multiple encounters: 16773


## Initial findings

- The dataset contains 101,766 hospital encounters from 71,518 unique patients.
- 16,773 patients have multiple encounters.
- No fully duplicated rows were found.
- Weight is missing in 96.86% of records.
- Readmissions within 30 days represent 11.16% of encounters.
- Training and test sets will be separated by patient to prevent overlap.
- Accuracy alone will not be sufficient to evaluate the model.

In [21]:
# Create a working copy and exclude weight
analysis_df = df.drop(columns=["weight"]).copy()

print("Original shape:", df.shape)
print("Working shape:", analysis_df.shape)

Original shape: (101766, 50)
Working shape: (101766, 49)


In [22]:
# Create a binary target for early readmission
analysis_df["readmitted_30_days"] = (
    analysis_df["readmitted"]
    .map({"<30": 1, ">30": 0, "NO": 0})
)

analysis_df["readmitted_30_days"].value_counts(dropna=False)

readmitted_30_days
0    90409
1    11357
Name: count, dtype: int64

In [23]:
# Compare hospital stay duration by early readmission outcome
stay_summary = analysis_df.groupby(
    "readmitted_30_days"
)["time_in_hospital"].agg(["count", "mean", "median"])

stay_summary.round(2)

,count,mean,median
readmitted_30_days,,,
0,90409,4.35,4.0
1,11357,4.77,4.0


## Hospital stay and early readmission

Encounters followed by readmission within 30 days had an average
hospital stay of 4.77 days, compared with 4.35 days for other encounters.
Both groups had a median stay of 4 days.

This descriptive difference does not establish causation or demonstrate
that hospital stay alone can accurately predict early readmission.

In [24]:
# Calculate early readmission rates by age group
age_summary = analysis_df.groupby("age").agg(
    encounters=("readmitted_30_days", "size"),
    early_readmissions=("readmitted_30_days", "sum"),
    readmission_rate=("readmitted_30_days", "mean")
)

age_summary["readmission_rate"] *= 100

age_summary.round(2)

,encounters,early_readmissions,readmission_rate
age,,,
[0-10),161,3,1.86
[10-20),691,40,5.79
[20-30),1657,236,14.24
[30-40),3775,424,11.23
[40-50),9685,1027,10.60
[50-60),17256,1668,9.67
[60-70),22483,2502,11.13
[70-80),26068,3069,11.77
[80-90),17197,2078,12.08


## Early readmission by age group

The 20–29 age group had the highest observed early readmission rate
(14.24%; 236 of 1,657 encounters).

The 70–79 age group had the largest number of early readmissions
(3,069), with a rate of 11.77%.

Readmission rates did not increase consistently with age.
These comparisons describe encounters, not unique patients,
and do not account for differences in other patient characteristics.